# LangChain + Ollama in Google Colab

From **raw chat** to **tool calling**, **retrieval over a document directory**, and a simple **multi-agent supervisor**.

This notebook is structured as a step-by-step tutorial:

1. **Environment setup** and configuration (including a non-localhost Ollama base URL).
2. **Raw chat** with an Ollama model via LangChain.
3. **Tool calling**: exposing a Python function that multiplies two numbers and letting the model call it.
4. **Retrieval-Augmented Generation (RAG)** over a local directory of documents.
5. **Multi-agent collaboration**: a supervisor agent that routes between a **calculator agent** and a **research agent**.

Along the way, cells are kept intentionally small and annotated so you can run, inspect, and modify each stage.


## What This Notebook Demonstrates

This notebook is an end-to-end tour of the **agentic stack** using LangChain and a local Ollama model: a raw chat call, then a model that can invoke a Python tool, then retrieval-augmented generation over your own documents, and finally a supervisor agent that routes questions to specialist sub-agents (and a clarifier-to-executor handoff).

**Course connection:** these are the exact building blocks — prompting, tool calling, RAG, and multi-agent orchestration — that the course's labs assemble into working systems. Notice as you go how each layer adds capability *and* a new failure surface: a tool the model might misuse, a retriever that might return nothing, a supervisor that might route to the wrong specialist. Evaluating and guarding those failure points is the other half of this course.

## 0. Environment & Dependencies

This section installs and imports the libraries we will use.

We rely on the modern LangChain package layout:

- `langchain` – core abstractions (prompts, runnables, etc.).
- `langchain-community` – community integrations (loaders, vector stores).
- `langchain-ollama` – integration for Ollama chat models and embeddings.
- `chromadb` – a lightweight vector store for local retrieval.

You should have an **Ollama server** running somewhere (local or remote) and a model (e.g. `llama3`) already pulled.
We will parameterize the base URL so it can be non-localhost.

In [ ]:
# If you re-run this notebook, you can safely re-run this cell.
# Installs are done with quiet mode to keep output readable.
!pip install -q langchain langchain-community langchain-ollama chromadb

print("Installed langchain, langchain-community, langchain-ollama, chromadb")

## 1. Core Imports and Configuration

Here we import the main classes we'll use and configure the connection to Ollama.

We will:

1. Import the chat model and embedding classes for Ollama.
2. Import LangChain core utilities (prompts, tools, runnables, parsers).
3. Import loaders, splitters, and vector stores for document retrieval.
4. Set up configuration variables for the **Ollama base URL** and **model name**.

You can override these via environment variables `OLLAMA_BASE_URL` and `OLLAMA_MODEL`.

In [ ]:
import os

# LangChain + Ollama
from langchain_ollama import ChatOllama, OllamaEmbeddings

# Core LangChain building blocks
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage, ToolMessage
from langchain_core.runnables import RunnableParallel, RunnablePassthrough

# Community integrations (loaders, vector stores)
from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain_community.vectorstores import Chroma

# Text splitting
from langchain.text_splitter import RecursiveCharacterTextSplitter

print("Imports complete.")

In [ ]:
# ---- Ollama connection configuration ----

# You can set these before running the notebook, e.g. in Colab:
#   %env OLLAMA_BASE_URL=http://your-remote-host:11434
#   %env OLLAMA_MODEL=llama3

OLLAMA_BASE_URL = os.environ.get("OLLAMA_BASE_URL", "http://localhost:11434")
OLLAMA_MODEL = os.environ.get("OLLAMA_MODEL", "llama3")

print(f"Using Ollama base URL: {OLLAMA_BASE_URL}")
print(f"Using Ollama model:    {OLLAMA_MODEL}")

### Sanity-check: instantiate a ChatOllama model

We now create a simple `ChatOllama` instance with a relatively low temperature for reproducible outputs.
If the base URL or model name are incorrect, you will see a connection error here.

In [ ]:
chat_model = ChatOllama(
    model=OLLAMA_MODEL,
    base_url=OLLAMA_BASE_URL,
    temperature=0.2,
)

print(chat_model)

## 2. A First Raw Query (No Tools, No Retrieval)

We will now send a **single chat prompt** through a very small LangChain pipeline:

1. A `ChatPromptTemplate` builds structured messages from a template.
2. `ChatOllama` generates a response.
3. `StrOutputParser` extracts the final text content.

This is the minimal building block upon which we will add tools and retrieval later.

In [ ]:
# 1. Build a simple chat prompt template
raw_chat_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a concise, technical assistant."),
    ("human", "{question}"),
])

# 2. Compose prompt -> model -> string parser into a runnable chain
raw_chat_chain = raw_chat_prompt | chat_model | StrOutputParser()

# 3. Invoke the chain with a concrete question
response = raw_chat_chain.invoke({
    "question": "In 3–4 sentences, what is LangChain and why is it useful?",
})

print(response)

### What just happened?

- We bound a **system message** to control the assistant's style.
- We parameterized the **human message** with a `{question}` placeholder.
- We piped messages into `ChatOllama`, then into `StrOutputParser` to get plain text.

Next, we will let the model **call a Python function** when it needs to multiply numbers.

## 3. Adding a Tool: Multiply Two Numbers

We now expose a small Python function as a **tool** so that the model can ask to call it.

Conceptually:

1. We write a regular Python function `multiply(a: int, b: int) -> int`.
2. We decorate it with `@tool` so that LangChain turns it into a tool schema.
3. We **bind** the tool to our `ChatOllama` model.
4. On each interaction:
   - The model may emit a **tool call** describing which tool to run and with what arguments.
   - Our Python code executes the tool and then feeds the result back to the model.

This gives the LLM safe, controlled access to deterministic computation.

In [ ]:
@tool
def multiply(a: int, b: int) -> int:
    """Multiply two integers and return their product."""
    return a * b

print("Tool name:", multiply.name)
print("Tool description:", multiply.description)

### YOUR TURN: Define Your Own Tool

**Before running: what do you expect and why?** You are about to add a second tool, `add`, alongside `multiply`. When the model later sees a question like "what is 7 plus 5?", what information do you think it uses to choose between the two tools — the function's *code*, or its *docstring and name*? Predict first; the answer shapes how you should write every tool you ever expose to an agent.

In [ ]:
# YOUR TURN: fill in the blanks (___) to define an addition tool.
# The docstring matters: it is the ONLY description the model sees when
# deciding whether to call this tool.

@tool
def add(a: int, b: int) -> int:
    """___"""   # fill in: a one-sentence description of what this tool does
    return ___    # fill in: the addition expression

print("Tool name:", add.name)
print("Tool description:", add.description)

In [ ]:
# Bind the multiply tool to the chat model.

tools = [multiply]
tool_chat_model = chat_model.bind_tools(tools)

print("Model is now aware of the following tools:")
for t in tools:
    print(" -", t.name)

### Single tool-call roundtrip

We will:

1. Ask the model a question that naturally requires multiplication.
2. Inspect the intermediate response for tool calls.
3. Execute the tool in Python.
4. Feed the result back to the model to obtain a final explanation.

In [ ]:
# Step 1: Ask a question that requires multiplication.
messages = [
    ("system", "You are a careful math assistant. If the question involves multiplying two numbers, use the multiply tool."),
    ("human", "What is 123 times 456? Show your reasoning."),
]

# Step 2: Let the model respond, possibly with tool calls.
first_ai_message = tool_chat_model.invoke(messages)
print("First AI message (may contain tool calls):\n")
print(first_ai_message)

# Step 3: Execute any tool calls we received.
tool_messages = []
if getattr(first_ai_message, "tool_calls", None):
    for call in first_ai_message.tool_calls:
        tool_name = call.get("name")
        tool_args = call.get("args", {})
        tool_id = call.get("id")

        if tool_name == multiply.name:
            result = multiply.invoke(tool_args)
            print(f"\nExecuted tool '{tool_name}' with args {tool_args} -> {result}")

            tool_messages.append(
                ToolMessage(
                    content=str(result),
                    tool_call_id=tool_id,
                )
            )

# Step 4: Send the tool results back to the model to get a final answer.
final_messages = messages + [first_ai_message] + tool_messages
final_ai_message = tool_chat_model.invoke(final_messages)

print("\nFinal answer:\n")
print(final_ai_message.content)

### YOUR TURN: A Two-Tool Roundtrip

**Before running: what do you expect and why?** Bind *both* tools (`multiply` and `add`) to the model and ask a question that needs addition. Do you predict the model will (a) call `add`, (b) call `multiply` with wrong intent, or (c) answer directly without any tool? What in your prompt or the tool docstrings would push it toward the right choice?

In [ ]:
# YOUR TURN: fill in the blanks (___) to run a tool roundtrip with BOTH tools bound.

both_tools = [multiply, ___]                     # fill in: the second tool object
two_tool_model = chat_model.bind_tools(both_tools)

messages_2 = [
    ("system", "You are a careful math assistant. Use a tool whenever it applies."),
    ("human", ___),                              # fill in: a question requiring addition
]

first_msg = two_tool_model.invoke(messages_2)
print("Tool calls requested:", getattr(first_msg, "tool_calls", None))

# Execute whichever tool was called, then send the result back for a final answer.
tool_msgs = []
for call in (getattr(first_msg, "tool_calls", None) or []):
    for t in both_tools:
        if call.get("name") == t.name:
            result = t.invoke(call.get("args", {}))
            print(f"Executed {t.name} -> {result}")
            tool_msgs.append(ToolMessage(content=str(result), tool_call_id=call.get("id")))

final_msg = two_tool_model.invoke(messages_2 + [first_msg] + tool_msgs)
print("\nFinal answer:\n", final_msg.content)

# After running: did the model choose the tool you predicted?

### Exercise

Try modifying the question to involve multiple operations (e.g., addition and multiplication) and see whether the model still chooses to call the `multiply` tool appropriately.

In more advanced systems, you would often **loop** this process until no further tool calls are produced.

## 4. Retrieval over a Directory of Documents (RAG)

Next we will add **retrieval-augmented generation (RAG)** so that the model can answer questions over a local document collection.

High-level pipeline:

1. Use `DirectoryLoader` to read files from a folder (here, plain text files for simplicity).
2. Split documents into overlapping chunks with `RecursiveCharacterTextSplitter`.
3. Embed chunks with `OllamaEmbeddings`.
4. Store embeddings in a local `Chroma` vector store and expose a `retriever`.
5. Build a RAG chain where the retriever feeds context into a prompt for the chat model.

In [ ]:
# Path to a directory of documents.
# In Colab, you might upload files into /content/docs or mount Google Drive.
DOCS_PATH = os.environ.get("DOCS_PATH", "/content/docs")
print(f"Document directory: {DOCS_PATH}")

In [ ]:
# Load all .txt files from the directory.
# You can adapt glob patterns or loader classes for PDFs, Markdown, etc.

loader = DirectoryLoader(
    DOCS_PATH,
    glob="**/*.txt",
    loader_cls=TextLoader,
    show_progress=True,
)

docs = loader.load()
print(f"Loaded {len(docs)} documents.")
if docs:
    print("Sample document metadata:", docs[0].metadata)

In [ ]:
# Split documents into overlapping chunks for dense retrieval.

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=200,
)

doc_chunks = text_splitter.split_documents(docs)
print(f"Created {len(doc_chunks)} chunks.")

In [ ]:
# Create an Ollama-based embedding model and a Chroma vector store.

embedding_model = OllamaEmbeddings(
    model=OLLAMA_MODEL,
    base_url=OLLAMA_BASE_URL,
)

vectorstore = Chroma.from_documents(
    documents=doc_chunks,
    embedding=embedding_model,
    collection_name="local-docs",
)

retriever = vectorstore.as_retriever(search_kwargs={"k": 4})
print("Vector store and retriever ready.")

### Building the RAG chain

We now define a prompt template that takes both a **question** and a **context** string. The context will be built by:

1. Running the query through the retriever.
2. Concatenating the retrieved text chunks.

We then form a chain:

```text
{question} ─▶ retriever ─▶ format_docs ─▶ prompt ─▶ ChatOllama ─▶ StrOutputParser
```

In [ ]:
def format_docs(docs):
    """Turn a list of Documents into a single context string."""
    return "\n\n".join(d.page_content for d in docs)

rag_prompt = ChatPromptTemplate.from_template(
    """You are a question-answering assistant.
Use the provided context to answer the user's question.
If the answer is not contained in the context, say you do not know.

Question:
{question}

Context:
{context}
"""
)

# Parallel branch: given a question, run retrieval and preserve the question.
rag_inputs = {
    "context": retriever | format_docs,
    "question": RunnablePassthrough(),
}

rag_chain = rag_inputs | rag_prompt | chat_model | StrOutputParser()

print("RAG chain constructed.")

In [ ]:
# If your DOCS_PATH is empty, this will likely respond with "I do not know".
# Once you populate DOCS_PATH with relevant .txt files, try questions specific to those files.

sample_question = "According to the documents, what topics are discussed most frequently?"
rag_answer = rag_chain.invoke(sample_question)
print(rag_answer)

### Exercise

1. Place a few `.txt` files into your `DOCS_PATH` directory.
2. Ask highly specific questions whose answers appear verbatim in those files.
3. Inspect `retriever.get_relevant_documents(question)` to see which chunks are selected.

## 5. Multi-Agent Collaboration via a Supervisor Pattern

We now combine the pieces into a simple **multi-agent system** using LangChain's tool-calling capabilities.

We will create:

1. A **calculator agent** that specializes in numeric reasoning and can call the `multiply` tool.
2. A **research agent** that specializes in answering questions over the local document collection using the RAG chain.
3. A **supervisor agent** (also an Ollama model) that chooses which sub-agent to call for a given user query.

We implement the pattern where **sub-agents are exposed as tools** to the supervisor. The supervisor decides which tool to call, we execute it, and then we feed the result back for a final user-facing answer.

In [ ]:
# 5.1 Calculator agent: wrap the tool-enabled chat model into a helper function.

def calculator_agent(question: str) -> str:
    """Use the tool-aware chat model (with multiply) to answer math questions."""
    base_messages = [
        ("system", "You are a calculator agent. Use the multiply tool whenever appropriate."),
        ("human", question),
    ]

    first_ai = tool_chat_model.invoke(base_messages)
    tool_messages = []

    if getattr(first_ai, "tool_calls", None):
        for call in first_ai.tool_calls:
            name = call.get("name")
            args = call.get("args", {})
            call_id = call.get("id")
            if name == multiply.name:
                result = multiply.invoke(args)
                tool_messages.append(
                    ToolMessage(content=str(result), tool_call_id=call_id)
                )

    final_messages = base_messages + [first_ai] + tool_messages
    final_ai = tool_chat_model.invoke(final_messages)
    return final_ai.content

print(calculator_agent("What is 37 * 91?"))

In [ ]:
# 5.2 Research agent: wrap the RAG chain.

def research_agent(question: str) -> str:
    """Use the RAG chain to answer questions over the local document collection."""
    return rag_chain.invoke(question)

print(research_agent("What kinds of topics appear in the documents?"))

### Expose sub-agents as tools for the supervisor

We now wrap `calculator_agent` and `research_agent` as `@tool`-decorated functions so the supervisor LLM can call them as tools.

In [ ]:
@tool
def calculator_subagent(query: str) -> str:
    """Use this for math-heavy questions, especially those involving multiplication."""
    return calculator_agent(query)


@tool
def research_subagent(query: str) -> str:
    """Use this for questions that should be answered from the local document collection."""
    return research_agent(query)

multiagent_tools = [calculator_subagent, research_subagent]
print("Multi-agent tools:")
for t in multiagent_tools:
    print(" -", t.name)

In [ ]:
# 5.3 Supervisor model that can call the sub-agent tools.

supervisor_model = ChatOllama(
    model=OLLAMA_MODEL,
    base_url=OLLAMA_BASE_URL,
    temperature=0.0,  # deterministic routing decisions
)

supervisor_with_tools = supervisor_model.bind_tools(multiagent_tools)

print("Supervisor model ready.")

In [ ]:
def supervisor_pipeline(user_query: str) -> str:
    """Route a user query through the supervisor to one of the sub-agents.

    The supervisor decides whether to call the calculator or research sub-agent
    (or neither) and we execute any resulting tool calls.
    """

    base_messages = [
        (
            "system",
            "You are a supervisor agent. You can delegate to two tools: "
            "`calculator_subagent` for arithmetic / numeric reasoning and "
            "`research_subagent` for questions about the local documents. "
            "Decide which tool is most appropriate, call it, and then synthesize "
            "a clear final answer for the user. If no tool is appropriate, answer directly.",
        ),
        ("human", user_query),
    ]

    first_ai = supervisor_with_tools.invoke(base_messages)
    tool_messages = []

    if getattr(first_ai, "tool_calls", None):
        for call in first_ai.tool_calls:
            name = call.get("name")
            args = call.get("args", {})
            call_id = call.get("id")

            if name == calculator_subagent.name:
                result = calculator_subagent.invoke(args)
            elif name == research_subagent.name:
                result = research_subagent.invoke(args)
            else:
                result = "(Unknown tool requested by model.)"

            tool_messages.append(
                ToolMessage(content=str(result), tool_call_id=call_id)
            )

    final_messages = base_messages + [first_ai] + tool_messages
    final_ai = supervisor_with_tools.invoke(final_messages)
    return final_ai.content


print("Example: math-heavy question routed to calculator agent\n")
print(supervisor_pipeline("Please compute 321 * 789 and explain your steps."))

In [ ]:
print("\nExample: knowledge question routed to research agent (if docs exist)\n")
print(supervisor_pipeline("Based on our documents, what are the main themes that appear?"))


## 6. Agent Handoff Example (Clarifier → Executor)

This section demonstrates a two-agent handoff workflow:
1. **Clarifier agent** rewrites ambiguous queries into canonical forms.
2. **Executor agent** solves the clarified query using existing tools or RAG.


In [ ]:

# Clarifier agent: rewrites or clarifies user intent
def clarifier_agent(query: str) -> str:
    prompt = ChatPromptTemplate.from_messages([
        ("system", "Rewrite the user's request into a precise, unambiguous task."),
        ("human", query),
    ])
    chain = prompt | chat_model | StrOutputParser()
    return chain.invoke({})

# Executor: uses supervisor pipeline defined earlier
def executor_agent(query: str) -> str:
    return supervisor_pipeline(query)

# Handoff logic
def clarifier_to_executor_pipeline(query: str) -> str:
    clarified = clarifier_agent(query)
    final = executor_agent(clarified)
    return final

# Example usage
print(clarifier_to_executor_pipeline("Can you figure out how many things there are if I have dozens of items?"))


### YOUR TURN: Stress-Test the Handoff

**Before running: what do you expect and why?** The clarifier agent rewrites vague requests before the executor acts on them. Pick a deliberately ambiguous query of your own (like the "dozens of items" example). Predict: what will the clarifier turn it into, and will the supervisor then route it to the calculator sub-agent or the research sub-agent? Write down both predictions.

In [ ]:
# YOUR TURN: fill in the blank (___) with your own deliberately vague query,
# then trace how it moves through the two-agent handoff.

vague_query = ___   # fill in: an ambiguous request, e.g. mixing math with fuzzy language

clarified = clarifier_agent(vague_query)
print("Clarified task:", clarified)

final_answer = executor_agent(clarified)
print("\nFinal answer:\n", final_answer)

# After running: did the clarifier's rewrite match your prediction? Did the
# supervisor route where you expected? If not, what one change to the clarifier's
# system prompt would fix the routing?

### Where to go from here

You now have a Colab notebook that demonstrates, end-to-end:

1. Connecting LangChain to an Ollama server at an arbitrary base URL.
2. Building a minimal chat pipeline.
3. Exposing a pure Python function (multiplication) as a callable tool for the model.
4. Creating a small RAG system over a directory of documents.
5. Composing specialized sub-agents behind a tool-calling **supervisor agent**.

Possible extensions:

- Add more tools to the calculator (e.g., division with error handling) or create new specialist agents.
- Swap in different Ollama models (e.g., smaller vs. larger, or reasoning-capable ones) for different agents.
- Replace `Chroma` with a different vector store, or add metadata filters to the retriever.
- Integrate streaming output or conversation memory for richer multi-turn interactions.

You can now adapt this notebook to your own workflows and infrastructure.